In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import *

### Scenario:
You're loading `sales_fact_unknown_member.csv` and joining it against `product_dim.csv`. A few sales reference `product_id`s that don't exist in the dimension table yet — likely because the product dimension load hasn't caught up with a newly launched product, or there's bad data upstream. Silently dropping these facts (via an inner join) would understate sales; leaving `null` product names would break downstream reports and BI dashboards that group by category. The standard warehousing fix is the **"Unknown Member" pattern**: orphan facts get mapped to a placeholder dimension value instead of null or being dropped.

**Problem:**

- Read both files with explicit schemas.
- Left join `sales_fact_unknown_member` to `product_dim` on `product_id` — do **not** use an inner join, since that would silently drop orphan facts.
- For rows where the join didn't find a match, replace the null `product_name` with `'Unknown Product'` and the null `category` with `'Unknown'` (use `coalesce()`).
- Separately, count how many fact rows were orphaned (i.e., had no matching `product_id` in the dimension) and print that count.
- Display the final joined output with columns `sale_id`, `product_id`, `product_name`, `category`, `quantity`, `sale_date`, ordered by `sale_id` ascending.

**product_dim.csv Schema**

| Column | Type |
| :--- | :--- |
| **product_id** | string |
| **product_name** | string |
| **category** | string |

**sales_fact_unknown_member.csv Schema**

| Column | Type |
| :--- | :--- |
| **sale_id** | string |
| **product_id** | string |
| **quantity** | int |
| **sale_date** | date |

**Expected Output — orphan count**

| orphan_count |
| :--- |
| 2 |

**Expected Output — final joined table**

| sale_id | product_id | product_name | category | quantity | sale_date |
| :--- | :--- | :--- | :--- | :--- | :--- |
| S001 | P001 | Widget A | Tools | 5 | 2024-09-01 |
| S002 | P002 | Widget B | Tools | 3 | 2024-09-01 |
| S003 | P099 | Unknown Product | Unknown | 2 | 2024-09-02 |
| S004 | P003 | Gadget C | Electronics | 1 | 2024-09-02 |
| S005 | P100 | Unknown Product | Unknown | 4 | 2024-09-03 |

In [0]:
schema_sales = StructType(
    [
        StructField("sale_id", StringType()),
        StructField("product_id", StringType()),
        StructField("quantity", IntegerType()),
        StructField("sale_date", DateType())
    ]
)
sales_df = spark.read.format("csv").option("header", True).schema(schema_sales).load("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/sales_fact_unknown_member.csv")

schema_products = StructType(
    [
        StructField("product_id", StringType()),
        StructField("product_name", StringType()),
        StructField("category", StringType())
    ]
)
products_df = spark.read.format("csv").option("header", True).schema(schema_products).load("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/product_dim.csv")

sales_prods_join_df = sales_df.join(products_df, "product_id", "left")
updated_sales_prod_df = sales_prods_join_df.withColumns(
    {
        "product_name": coalesce(col("product_name"), lit("Unknown Product")),
        "category": coalesce(col("category"), lit("Unknown"))
    }
)

orphan_count = updated_sales_prod_df.filter(col("product_name") == "Unknown Product").count()
print(f"orphan_count: {orphan_count}")

updated_sales_prod_df.select("sale_id", "product_id", "product_name", "category", "quantity", "sale_date").orderBy("sale_id").show()

### Scenario:
You're given `customer_updates_dupes.csv` — a batch of CDC-style (change data capture) update records from an upstream source. The same `customer_id` can appear multiple times in one batch if the customer's record was updated more than once before your pipeline ran. If you load all of them as-is, you'll double-count customers and your dimension table will have stale duplicate versions sitting alongside the correct one. You need to keep only the **most recent** record per customer before loading.

**Problem:**

- Read the file with an explicit schema.
- For each `customer_id`, identify the single most recent record based on `updated_at` (use `row_number()` over an appropriately ordered window — do not use `dense_rank()` here, since you specifically want exactly one row per customer even if timestamps tie).
- Keep only that latest row per customer, discard the rest.
- Order the output by `customer_id` ascending.

**Schema**

| Column | Type |
| :--- | :--- |
| **customer_id** | string |
| **customer_name** | string |
| **email** | string |
| **updated_at** | timestamp |

**Expected Output**

| customer_id | customer_name | email | updated_at |
| :--- | :--- | :--- | :--- |
| C001 | Alice J. | alice.j@x.com | 2024-10-01 09:30:00 |
| C002 | Bob Singh | bob@x.com | 2024-10-01 08:15:00 |
| C003 | Carol Mehta | carol.mehta@x.com | 2024-10-01 10:00:00 |
| C004 | David Kim | david@x.com | 2024-10-01 09:00:00 |

In [0]:
schema_cust = StructType(
    [
        StructField("customer_id", StringType()),
        StructField("customer_name", StringType()),
        StructField("email", StringType()),
        StructField("updated_at", TimestampType()),
    ]
)

cust_df = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema_cust)
    .load(
        "/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/customer_updates_dupes.csv"
    )
)

window_logic = Window.partitionBy(col("customer_id")).orderBy(col("updated_at").desc())

cust_latest_identity_df = cust_df.withColumn(
    "priority", row_number().over(window_logic)
)
cust_latest_identity_df.filter(col("priority") == 1).select(
    "customer_id", "customer_name", "email", "updated_at"
).show()

### Scenario:
You're joining two sources stored in two different formats — a common real-world pattern in a data lake. `product_catalog.parquet` is your product dimension, stored as Parquet (columnar, schema embedded, no parsing needed). `order_events_drift.json` is a raw order event feed — and it has **schema drift**: some records include a `discount` field, others simply omit it entirely (not `null` — the key doesn't exist in that record at all). Spark will still union these into one schema when reading, but any row missing the field will get a `null` there, which you need to handle before doing arithmetic on it.

**Problem:**

- Read `product_catalog.parquet` using the Parquet reader — note that Parquet is self-describing, so you do **not** need to define a schema manually; just load it and confirm the inferred types with `.printSchema()`.
- Read `order_events_drift.json` — again check `.printSchema()` first and notice how Spark handled the missing `discount` field for the records that didn't have it.
- Join the orders to the product catalog on `product_id`.
- Compute `revenue` = (`qty` × `price`) − `discount`, treating a missing/null `discount` as `0` (use `coalesce()`).
- Compute `profit` = `revenue` − (`qty` × `unit_cost`).
- Select `order_id`, `product_name`, `category`, `revenue`, `profit`, ordered by `order_id` ascending.

**Expected Output**

| order_id | product_name | category | revenue | profit |
| :--- | :--- | :--- | :--- | :--- |
| O1 | Widget A | Tools | 28.0 | 13.0 |
| O2 | Widget B | Tools | 25.0 | 13.0 |
| O3 | Gadget C | Electronics | 75.0 | 35.0 |
| O4 | Widget A | Tools | 50.0 | 25.0 |

In [0]:
product_catalog_df = spark.read.format("parquet").load("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/product_catalog.parquet")

"""
product_catalog_df.printSchema()
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_cost: double (nullable = true)
"""

orders_df = spark.read.format("json").load("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/order_events_drift.json")

"""
orders_evt_json.printSchema()
root
 |-- discount: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- qty: long (nullable = true)
"""

product_orders_df = orders_df.join(product_catalog_df, "product_id", "inner").withColumns(
    {
        "revenue": (col("qty") * col("price")) - coalesce(col("discount"), lit(0.0).cast(DoubleType())),
        "profit": (col("revenue")) - (col("qty") * col("unit_cost"))
    }
)
product_orders_df.select("order_id", "product_name", "category", "revenue", "profit").orderBy(col("order_id").asc()).show()